In [3]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    temperature=0,
    groq_api_key = "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",
    model_name = "llama-3.3-70b-versatile"

)
result = llm.invoke("Who is lord Ram?")
print(result.content)

Lord Rama is a major deity in Hinduism and is considered one of the most important figures in Hindu mythology. He is the seventh avatar (incarnation) of the god Vishnu and is revered as a symbol of righteousness, duty, and virtue.

According to the Hindu epic, the Ramayana, Lord Rama was a king of Ayodhya, a city in ancient India. He was born to King Dasharatha and Queen Kausalya, and was the eldest of four brothers, including Lakshmana, Bharata, and Shatrughna.

The story of Lord Rama is as follows:

* Lord Rama was exiled to the forest for 14 years by his stepmother, Queen Kaikeyi, who wanted her own son, Bharata, to become the king.
* During his exile, Lord Rama's wife, Sita, was abducted by the demon king Ravana, who took her to his kingdom in Lanka (modern-day Sri Lanka).
* Lord Rama, along with his brother Lakshmana and the monkey god Hanuman, embarked on a journey to rescue Sita and defeat Ravana.
* After a long and arduous battle, Lord Rama defeated Ravana and rescued Sita, and

In [7]:
!pip install pypdf
!pip install chromadb
!pip install sentence_transformers

In [5]:
import os

In [5]:
!pip install --upgrade "langchain<1.0" "langchain-community<1.0" "langchain-core<1.0" "langchain-huggingface<1.0"
!pip install transformers

  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_community-0.4-py3-none-any.whl.metadata (3.0 kB)


In [11]:
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 19.2 MB/s  0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.46.4
    Uninstalling pydantic_core-2.46.4:
      Successfully uninstalled pydantic_core-2.46.4
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.13.4
    Uninstalling pydantic-2.13.4:
      Successfully uninstalled pydantic-2.13.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pydantic]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph 1.2.6 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.3.86 which is incompatible.
langchain-classic 1.0.8 requires langchain-core<2.0.0,>=1.4.4, but you have langchain-core 0.3.86 which is incompatible.
langchain-classic 1.0.8 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.1

In [9]:
import os
import gradio as gr
from langchain.embeddings import HuggingFaceEmbeddings

from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_community.vectorstores import Chroma

from langchain.chains import RetrievalQA # Changed to import RetrievalQA
# Removed: from langchain.chains.retrieval import create_retrieval_chain
# Removed: from langchain.chains.combine_documents import create_stuff_documents_chain

from langchain_core.prompts import PromptTemplate

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq

def initialize_llm():
  llm = ChatGroq(
    temperature=0,
    groq_api_key = "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",
    model_name = "llama-3.3-70b-versatile"

)
  return llm

def create_vector_db():
  loader = DirectoryLoader("/content/data", glob="*.pdf", loader_cls=PyPDFLoader)
  documents = loader.load()
  text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
  texts = text_splitter.split_documents(documents)
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
  vector_db = Chroma.from_documents(texts, embeddings, persist_directory=
                             './chroma_db')
  vector_db.persist()
  print("ChromaDB created and data saved")
  return vector_db

def setup_qa_chain(vector_db, llm):
  retriever = vector_db.as_retriever(search_kwargs={"k": 2})
  promt_template = """You are a compassionate mental health chatbot.Respond thoughfully to the following question
  {context}
  User: {question}
  Chatbot:"""
  PROMPT = PromptTemplate(template=promt_template, input_variables=["context", "question"])
  # Reverted to RetrievalQA.from_chain_type for langchain < 1.0 compatibility
  qa_chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever, chain_type_kwargs={"prompt": PROMPT})
  return qa_chain

print("Initializing Chatbot.........")

llm = initialize_llm()
db_path = "/content/chroma_db"

if not os.path.exists(db_path):
  vector_db = create_vector_db()
else:
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
  vector_db = Chroma(persist_directory=db_path, embedding_function=embeddings)

qa_chain = setup_qa_chain(vector_db, llm)

def chatbot_response(message, history):
    if not message.strip():
        return "Please provide a valid input."

    response = qa_chain.run(message)
    return response

with gr.Blocks(theme = 'Respari/Shiki@1.2.1') as app:
  chatbot = gr.ChatInterface(fn = chatbot_response, title = "Mental Health Chatbot")

app.launch()

Initializing Chatbot.........


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/tmp/ipykernel_21542/623361172.py:69: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme = 'Respari/Shiki@1.2.1') as app:
/usr/local/lib/python3.12/dist-packages/gradio/blocks.py:1143: UserWarning: Cannot load Respari/Shiki@1.2.1. Caught Exception: The space Respari/Shiki does not exist
  warnings.warn(f"Cannot load {theme}. Caught Exception: {str(e)}")
/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d937e614ea3d7e7165.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [8]:
import os
import gradio as gr
from langchain.embeddings import HuggingFaceEmbeddings

from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_community.vectorstores import Chroma

from langchain.chains import RetrievalQA # Changed to import RetrievalQA
# Removed: from langchain.chains.retrieval import create_retrieval_chain
# Removed: from langchain.chains.combine_documents import create_stuff_documents_chain

from langchain_core.prompts import PromptTemplate

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq

def initialize_llm():
  llm = ChatGroq(
    temperature=0,
    groq_api_key = "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",
    model_name = "llama-3.3-70b-versatile"

)
  return llm

def create_vector_db():
  loader = DirectoryLoader("/content/data", glob="*.pdf", loader_cls=PyPDFLoader)
  documents = loader.load()
  text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
  texts = text_splitter.split_documents(documents)
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
  vector_db = Chroma.from_documents(texts, embeddings, persist_directory=
                             './chroma_db')
  vector_db.persist()
  print("ChromaDB created and data saved")
  return vector_db

def setup_qa_chain(vector_db, llm):
  retriever = vector_db.as_retriever(search_kwargs={"k": 2})
  promt_template = """You are a compassionate mental health chatbot.Respond thoughfully to the following question
  {context}
  User: {question}
  Chatbot:"""
  PROMPT = PromptTemplate(template=promt_template, input_variables=["context", "question"])
  # Reverted to RetrievalQA.from_chain_type for langchain < 1.0 compatibility
  qa_chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever, chain_type_kwargs={"prompt": PROMPT})
  return qa_chain

def main():

  print("Initializing Chatbot.........")

  llm = initialize_llm()
  db_path = "/content/chroma_db"

  if not os.path.exists(db_path):
    vector_db = create_vector_db()
  else:
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vector_db = Chroma(persist_directory=db_path, embedding_function=embeddings)

  qa_chain = setup_qa_chain(vector_db, llm)

  while True:
    query = input("\nHuman: ")
    if query.lower() == "exit":
      print("Chatbot: Take Care of yourself, Goodbye!")
      break
    response = qa_chain.run(query)
    print(f"Chatbot: {response}")

if __name__ == "__main__":
  main()


Initializing Chatbot.........


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Human: i'm stressed
Chatbot: I'm so glad you reached out and shared how you're feeling. It takes a lot of courage to acknowledge and express our emotions, especially when we're feeling stressed. 

It sounds like you might be experiencing some emotional distress, and I want you to know that I'm here to listen and support you. The vulnerability-stress model you mentioned earlier suggests that ongoing stress can have a significant impact on our mental health, and it's possible that you might be feeling some of those effects.

Can you tell me more about what's been going on and how you've been feeling lately? Is there something specific that's causing you stress, or is it more of a general feeling of overwhelm? Sometimes talking about our emotions and concerns can help us feel a bit better and gain some perspective.

Also, I want to acknowledge that it's completely normal to feel stressed, and it doesn't necessarily mean that you're experiencing depression. However, if you are struggling 